In [61]:
# Libraries needed to extract data from excel file and manipulate with pandas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from openpyxl import load_workbook
import re

In [9]:
def inspect_sheet(filepath, sheet_name, max_rows=80, max_cols=12):
    wb = load_workbook(filepath, data_only=True)
    ws = wb[sheet_name]

    print("Hojas disponibles:", wb.sheetnames)
    print(f"\nInspeccionando hoja: {sheet_name}\n")

    for r in range(1, max_rows + 1):
        values = []
        for c in range(1, max_cols + 1):
            val = ws.cell(row=r, column=c).value
            values.append(val)

        # imprimir solo filas que tengan al menos algo
        if any(v is not None and str(v).strip() != "" for v in values):
            print(f"Fila {r}: {values}")




In [19]:
# Cambia esto por tu archivo real
filepath = "trimestral_bcn_contractes.xlsx"

In [20]:
wb = load_workbook(filepath, data_only=True)
print(wb.sheetnames)

['2025', '2024', '2023', '2022', '2021', '2020', '2019', '2018', '2017', '2016', '2015', '2014', '2013', '2012', '2011', '2010', '2009', '2008', '2007', '2006', '2005', '2004', '2003', '2002', '2001', '2000']


In [21]:
inspect_sheet(filepath, sheet_name='2025')

Hojas disponibles: ['2025', '2024', '2023', '2022', '2021', '2020', '2019', '2018', '2017', '2016', '2015', '2014', '2013', '2012', '2011', '2010', '2009', '2008', '2007', '2006', '2005', '2004', '2003', '2002', '2001', '2000']

Inspeccionando hoja: 2025

Fila 1: ['Evolució trimestral del mercat de lloguer a Barcelona', None, None, None, None, None, None, None, None, None, None, None]
Fila 2: ['Nombre de contractes', None, None, None, None, None, None, None, None, None, None, None]
Fila 4: [None, None, 2025, None, None, None, None, '2025 (acumulat)', None, None, None, None]
Fila 5: ['Codi', None, 'I', 'II', 'III', 'IV', None, 'I', 'II', 'III', 'IV', None]
Fila 6: [None, 'Barcelona', 7615, 7411, 7855, None, None, 7615, 15026, 22881, None, None]
Fila 8: [None, 'Districtes municipals (1)', None, None, None, None, None, None, None, None, None, None]
Fila 9: [1, 'Ciutat Vella', 738, 659, 648, None, None, 738, 1397, 2045, None, None]
Fila 10: [2, 'Eixample', 1407, 1430, 1514, None, None, 140

In [35]:
def clean_text(value):
    """Limpia espacios extra y convierte None en string vacío."""
    if value is None:
        return ""
    return " ".join(str(value).strip().split())


def parse_nullable_int(value):
    """
    Convierte a int si es posible.
    Si viene vacío, 'n.d.' u otro valor no numérico, devuelve None.
    """
    if value is None:
        return None

    text = str(value).strip()

    if text == "":
        return None

    if text.lower() in {"n.d.", "nd", "n.d", "na", "n/a", "-"}:
        return None

    try:
        return int(float(text))
    except ValueError:
        return None


def extract_barris_one_sheet(filepath, sheet_name):
    """
    Extrae una hoja trimestral (ej. '2025') del Excel de Barcelona
    y devuelve una list[dict] a nivel barrio-año-trimestre.
    """
    wb = load_workbook(filepath, data_only=True)
    ws = wb[sheet_name]

    rows = list(ws.iter_rows(values_only=True))

    barris_row_idx = None

    # localizar la fila donde empieza la sección Barris
    for i, row in enumerate(rows):
        second_cell = clean_text(row[1]) if len(row) > 1 else ""
        if second_cell.startswith("Barris"):
            barris_row_idx = i
            break

    if barris_row_idx is None:
        raise ValueError(f"No encontré la fila 'Barris' en la hoja {sheet_name}.")

    year = int(sheet_name)

    # col 2 = I, col 3 = II, col 4 = III, col 5 = IV
    quarter_cols = {
        2: 1,
        3: 2,
        4: 3,
        5: 4,
    }

    result = []

    for i in range(barris_row_idx + 1, len(rows)):
        row = rows[i]

        code = row[0] if len(row) > 0 else None
        neighborhood = clean_text(row[1]) if len(row) > 1 else ""

        # fin del bloque de barrios
        if code is None or neighborhood == "":
            break

        if not isinstance(code, int):
            break

        for col_idx, quarter in quarter_cols.items():
            value = row[col_idx] if col_idx < len(row) else None

            result.append({
                "neighborhood_code": code,
                "neighborhood": neighborhood,
                "year": year,
                "quarter": quarter,
                "num_contracts": parse_nullable_int(value),
            })

    return result


def get_year_sheets(filepath, start_year=2014, end_year=2025):
    """
    Devuelve las hojas cuyo nombre sea un año dentro del rango pedido.
    """
    wb = load_workbook(filepath, data_only=True)

    valid_sheets = []
    for sheet_name in wb.sheetnames:
        if str(sheet_name).isdigit():
            year = int(sheet_name)
            if start_year <= year <= end_year:
                valid_sheets.append(sheet_name)

    return sorted(valid_sheets, key=int)


def extract_barris_all_years(filepath, start_year=2014, end_year=2025):
    """
    Concatena todas las hojas entre start_year y end_year
    en una sola list[dict].
    """
    year_sheets = get_year_sheets(filepath, start_year, end_year)

    all_rows = []

    for sheet_name in year_sheets:
        sheet_rows = extract_barris_one_sheet(filepath, sheet_name)
        all_rows.extend(sheet_rows)

    all_rows.sort(key=lambda x: (x["year"], x["neighborhood_code"], x["quarter"]))

    return all_rows


def summarize_panel(rows):
    """
    Resumen simple para validar la extracción.
    """
    years = sorted({row["year"] for row in rows})
    neighborhoods = sorted({row["neighborhood_code"] for row in rows})

    print("Años incluidos:", years)
    print("Cantidad de años:", len(years))
    print("Cantidad de barrios únicos:", len(neighborhoods))
    print("Total registros:", len(rows))

    print("\nRegistros por año:")
    counts_by_year = {}
    for row in rows:
        y = row["year"]
        counts_by_year[y] = counts_by_year.get(y, 0) + 1

    for year in sorted(counts_by_year):
        print(year, "->", counts_by_year[year])

# Ejecucion

In [36]:
contracts_panel = extract_barris_all_years(filepath, start_year=2014, end_year=2025)

summarize_panel(contracts_panel)

print("\nPrimeros 20 registros:")
for row in contracts_panel[:20]:
    print(row)

print("\nÚltimos 20 registros:")
for row in contracts_panel[-20:]:
    print(row)

Años incluidos: [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Cantidad de años: 12
Cantidad de barrios únicos: 73
Total registros: 3504

Registros por año:
2014 -> 292
2015 -> 292
2016 -> 292
2017 -> 292
2018 -> 292
2019 -> 292
2020 -> 292
2021 -> 292
2022 -> 292
2023 -> 292
2024 -> 292
2025 -> 292

Primeros 20 registros:
{'neighborhood_code': 1, 'neighborhood': 'el Raval', 'year': 2014, 'quarter': 1, 'num_contracts': 356}
{'neighborhood_code': 1, 'neighborhood': 'el Raval', 'year': 2014, 'quarter': 2, 'num_contracts': 409}
{'neighborhood_code': 1, 'neighborhood': 'el Raval', 'year': 2014, 'quarter': 3, 'num_contracts': 423}
{'neighborhood_code': 1, 'neighborhood': 'el Raval', 'year': 2014, 'quarter': 4, 'num_contracts': 395}
{'neighborhood_code': 2, 'neighborhood': 'el Barri Gòtic', 'year': 2014, 'quarter': 1, 'num_contracts': 135}
{'neighborhood_code': 2, 'neighborhood': 'el Barri Gòtic', 'year': 2014, 'quarter': 2, 'num_contracts': 169}
{'neighborhood_code

# Validación

In [37]:
def validate_contracts_panel(rows):
    """
    Valida estructura y completitud básica del panel.
    Espera una list[dict] con:
    neighborhood_code, neighborhood, year, quarter, num_contracts
    """

    print("========== VALIDACIÓN PANEL CONTRACTS ==========\n")

    # -------------------------------------------------
    # 1) Duplicados por clave natural
    # -------------------------------------------------
    seen = set()
    duplicates = []

    for row in rows:
        key = (row["neighborhood_code"], row["year"], row["quarter"])
        if key in seen:
            duplicates.append(key)
        else:
            seen.add(key)

    print("1. Duplicados por (neighborhood_code, year, quarter):")
    print("   Total duplicados:", len(duplicates))
    if duplicates[:10]:
        print("   Ejemplos:", duplicates[:10])
    print()

    # -------------------------------------------------
    # 2) Combinaciones esperadas vs observadas
    # -------------------------------------------------
    neighborhood_codes = sorted({row["neighborhood_code"] for row in rows})
    years = sorted({row["year"] for row in rows})
    quarters = [1, 2, 3, 4]

    expected_keys = set()
    for code in neighborhood_codes:
        for year in years:
            for quarter in quarters:
                expected_keys.add((code, year, quarter))

    observed_keys = {
        (row["neighborhood_code"], row["year"], row["quarter"])
        for row in rows
    }

    missing_keys = sorted(expected_keys - observed_keys)

    print("2. Combinaciones esperadas vs observadas:")
    print("   Esperadas:", len(expected_keys))
    print("   Observadas:", len(observed_keys))
    print("   Faltantes:", len(missing_keys))
    if missing_keys[:10]:
        print("   Ejemplos faltantes:", missing_keys[:10])
    print()

    # -------------------------------------------------
    # 3) Missing values en num_contracts por año-quarter
    # -------------------------------------------------
    missing_by_year_quarter = {}

    for row in rows:
        yq = (row["year"], row["quarter"])
        if yq not in missing_by_year_quarter:
            missing_by_year_quarter[yq] = 0

        if row["num_contracts"] is None:
            missing_by_year_quarter[yq] += 1

    print("3. Missing num_contracts por year-quarter:")
    for yq in sorted(missing_by_year_quarter):
        print(f"   {yq}: {missing_by_year_quarter[yq]}")
    print()

    # -------------------------------------------------
    # 4) Consistencia code -> neighborhood name
    # -------------------------------------------------
    names_by_code = {}

    for row in rows:
        code = row["neighborhood_code"]
        name = row["neighborhood"]

        if code not in names_by_code:
            names_by_code[code] = set()

        names_by_code[code].add(name)

    inconsistent_names = {
        code: names
        for code, names in names_by_code.items()
        if len(names) > 1
    }

    print("4. Consistencia neighborhood_code -> neighborhood:")
    print("   Códigos con más de un nombre:", len(inconsistent_names))
    if inconsistent_names:
        for code, names in list(inconsistent_names.items())[:10]:
            print(f"   Código {code}: {sorted(names)}")
    print()

    # -------------------------------------------------
    # 5) Resumen final
    # -------------------------------------------------
    total_rows = len(rows)
    total_missing = sum(1 for row in rows if row["num_contracts"] is None)

    print("5. Resumen final:")
    print("   Total filas:", total_rows)
    print("   Total barrios:", len(neighborhood_codes))
    print("   Total años:", len(years))
    print("   Total missing num_contracts:", total_missing)

    print("\n========== FIN VALIDACIÓN ==========")

In [38]:
validate_contracts_panel(contracts_panel)

========== VALIDACIÓN PANEL CONTRACTS ==========

1. Duplicados por (neighborhood_code, year, quarter):
   Total duplicados: 0

2. Combinaciones esperadas vs observadas:
   Esperadas: 3504
   Observadas: 3504
   Faltantes: 0

3. Missing num_contracts por year-quarter:
   (2014, 1): 0
   (2014, 2): 0
   (2014, 3): 1
   (2014, 4): 1
   (2015, 1): 0
   (2015, 2): 0
   (2015, 3): 0
   (2015, 4): 0
   (2016, 1): 0
   (2016, 2): 0
   (2016, 3): 0
   (2016, 4): 0
   (2017, 1): 0
   (2017, 2): 0
   (2017, 3): 0
   (2017, 4): 0
   (2018, 1): 0
   (2018, 2): 0
   (2018, 3): 0
   (2018, 4): 0
   (2019, 1): 0
   (2019, 2): 0
   (2019, 3): 0
   (2019, 4): 0
   (2020, 1): 0
   (2020, 2): 0
   (2020, 3): 0
   (2020, 4): 0
   (2021, 1): 0
   (2021, 2): 0
   (2021, 3): 0
   (2021, 4): 0
   (2022, 1): 0
   (2022, 2): 0
   (2022, 3): 0
   (2022, 4): 0
   (2023, 1): 0
   (2023, 2): 0
   (2023, 3): 0
   (2023, 4): 0
   (2024, 1): 0
   (2024, 2): 0
   (2024, 3): 0
   (2024, 4): 0
   (2025, 1): 0
   (2025, 2

# Funcion con PANDAS

In [39]:
import pandas as pd
from openpyxl import load_workbook


# =========================================================
# 1. UTILIDADES
# =========================================================

def clean_text(value):
    """Limpia espacios extra y convierte None en string vacío."""
    if value is None:
        return ""
    return " ".join(str(value).strip().split())


def parse_nullable_number(value, number_type="float"):
    """
    Convierte un valor a int o float.
    Si viene vacío, 'n.d.' u otro valor no numérico, devuelve None.
    """
    if value is None:
        return None

    # si openpyxl ya lo leyó como número
    if isinstance(value, (int, float)):
        if number_type == "int":
            return int(value)
        return float(value)

    text = str(value).strip()

    if text == "":
        return None

    if text.lower() in {"n.d.", "nd", "n.d", "na", "n/a", "-"}:
        return None

    # por si vienen comas decimales
    text = text.replace(",", ".")

    try:
        if number_type == "int":
            return int(float(text))
        return float(text)
    except ValueError:
        return None


def get_year_sheets(filepath, start_year=2014, end_year=2025):
    """
    Devuelve las hojas cuyo nombre sea un año dentro del rango pedido.
    """
    wb = load_workbook(filepath, data_only=True)

    valid_sheets = []
    for sheet_name in wb.sheetnames:
        if str(sheet_name).isdigit():
            year = int(sheet_name)
            if start_year <= year <= end_year:
                valid_sheets.append(sheet_name)

    return sorted(valid_sheets, key=int)


# =========================================================
# 2. EXTRACCIÓN GENÉRICA DE UNA HOJA
# =========================================================

def extract_barris_one_sheet_df(filepath, sheet_name, value_col_name, number_type="float"):
    """
    Extrae una hoja trimestral (ej. '2025') y devuelve un DataFrame
    a nivel barrio-año-trimestre.

    Parámetros
    ----------
    filepath : str
    sheet_name : str
    value_col_name : str
        Nombre de la variable resultado, por ejemplo:
        'num_contracts', 'avg_rent', 'avg_rent_m2', 'avg_surface'
    number_type : str
        'int' o 'float'
    """
    wb = load_workbook(filepath, data_only=True)
    ws = wb[sheet_name]

    rows = list(ws.iter_rows(values_only=True))

    barris_row_idx = None

    # localizar la fila donde empieza la sección Barris
    for i, row in enumerate(rows):
        second_cell = clean_text(row[1]) if len(row) > 1 else ""
        if second_cell.startswith("Barris"):
            barris_row_idx = i
            break

    if barris_row_idx is None:
        raise ValueError(f"No encontré la fila 'Barris' en la hoja {sheet_name}.")

    year = int(sheet_name)

    # columnas trimestrales normales
    # col 2 = I, col 3 = II, col 4 = III, col 5 = IV
    quarter_cols = {
        2: 1,
        3: 2,
        4: 3,
        5: 4,
    }

    result = []

    for i in range(barris_row_idx + 1, len(rows)):
        row = rows[i]

        code = row[0] if len(row) > 0 else None
        neighborhood = clean_text(row[1]) if len(row) > 1 else ""

        # fin del bloque de barrios
        if code is None or neighborhood == "":
            break

        if not isinstance(code, int):
            break

        for col_idx, quarter in quarter_cols.items():
            value = row[col_idx] if col_idx < len(row) else None

            result.append({
                "neighborhood_code": code,
                "neighborhood": neighborhood,
                "year": year,
                "quarter": quarter,
                value_col_name: parse_nullable_number(value, number_type=number_type),
            })

    df = pd.DataFrame(result)

    # orden prolijo
    df = df.sort_values(["year", "neighborhood_code", "quarter"]).reset_index(drop=True)

    return df


# =========================================================
# 3. EXTRACCIÓN GENÉRICA DE TODAS LAS HOJAS
# =========================================================

def extract_barris_all_years_df(
    filepath,
    value_col_name,
    number_type="float",
    start_year=2014,
    end_year=2025
):
    """
    Concatena todas las hojas entre start_year y end_year
    en un solo DataFrame.
    """
    year_sheets = get_year_sheets(filepath, start_year, end_year)

    dfs = []

    for sheet_name in year_sheets:
        df_sheet = extract_barris_one_sheet_df(
            filepath=filepath,
            sheet_name=sheet_name,
            value_col_name=value_col_name,
            number_type=number_type
        )
        dfs.append(df_sheet)

    df = pd.concat(dfs, ignore_index=True)
    df = df.sort_values(["year", "neighborhood_code", "quarter"]).reset_index(drop=True)

    return df


# =========================================================
# 4. VALIDACIÓN GENÉRICA DEL PANEL
# =========================================================

def validate_panel_df(df, value_col_name):
    """
    Valida estructura del panel usando pandas.
    """
    print(f"========== VALIDACIÓN PANEL: {value_col_name} ==========\n")

    # 1. Duplicados
    dup_mask = df.duplicated(subset=["neighborhood_code", "year", "quarter"], keep=False)
    dup_count = dup_mask.sum()

    print("1. Duplicados por (neighborhood_code, year, quarter):")
    print("   Total duplicados:", int(dup_count))
    if dup_count > 0:
        print(df.loc[dup_mask, ["neighborhood_code", "neighborhood", "year", "quarter"]].head(10))
    print()

    # 2. Cobertura esperada
    years = sorted(df["year"].dropna().unique().tolist())
    neighborhoods = sorted(df["neighborhood_code"].dropna().unique().tolist())
    quarters = [1, 2, 3, 4]

    expected = len(years) * len(neighborhoods) * len(quarters)
    observed = len(df)

    print("2. Combinaciones esperadas vs observadas:")
    print("   Esperadas:", expected)
    print("   Observadas:", observed)
    print("   Faltantes:", expected - observed)
    print()

    # 3. Missing por year-quarter
    missing_summary = (
        df.assign(is_missing=df[value_col_name].isna())
          .groupby(["year", "quarter"], as_index=False)["is_missing"]
          .sum()
          .rename(columns={"is_missing": "missing_count"})
    )

    print(f"3. Missing {value_col_name} por year-quarter:")
    print(missing_summary.to_string(index=False))
    print()

    # 4. Consistencia code -> neighborhood
    names_per_code = (
        df.groupby("neighborhood_code")["neighborhood"]
          .nunique()
          .reset_index(name="n_names")
    )

    inconsistent = names_per_code[names_per_code["n_names"] > 1]

    print("4. Consistencia neighborhood_code -> neighborhood:")
    print("   Códigos con más de un nombre:", len(inconsistent))
    if len(inconsistent) > 0:
        print(inconsistent.head(10).to_string(index=False))
    print()

    # 5. Resumen final
    total_missing = int(df[value_col_name].isna().sum())

    print("5. Resumen final:")
    print("   Total filas:", len(df))
    print("   Total barrios:", len(neighborhoods))
    print("   Total años:", len(years))
    print(f"   Total missing {value_col_name}:", total_missing)

    print(f"\n========== FIN VALIDACIÓN: {value_col_name} ==========")




In [40]:
# 5. EJEMPLOS DE USO


# 1) Número de contratos
contracts_panel_df = extract_barris_all_years_df(
    filepath="trimestral_bcn_contractes.xlsx",
    value_col_name="num_contracts",
    number_type="int",
    start_year=2014,
    end_year=2025
)



In [41]:
contracts_panel_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3504 entries, 0 to 3503
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   neighborhood_code  3504 non-null   int64  
 1   neighborhood       3504 non-null   object 
 2   year               3504 non-null   int64  
 3   quarter            3504 non-null   int64  
 4   num_contracts      3429 non-null   float64
dtypes: float64(1), int64(3), object(1)
memory usage: 137.0+ KB


In [42]:
# 2) Alquiler medio
avg_rent_df = extract_barris_all_years_df(
    filepath="trimestral_bcn_lloguer.xlsx",
    value_col_name="avg_rent",
    number_type="float",
    start_year=2014,
    end_year=2025
)

# 3) Alquiler medio por m2
avg_rent_m2_df = extract_barris_all_years_df(
    filepath="trimestral_bcn_lloguer_m2.xlsx",
    value_col_name="avg_rent_m2",
    number_type="float",
    start_year=2014,
    end_year=2025
)

# 4) Superficie media
avg_surface_df = extract_barris_all_years_df(
    filepath="trimestral_bcn_sup.xlsx",
    value_col_name="avg_surface",
    number_type="float",
    start_year=2014,
    end_year=2025
)



In [47]:
# info information for each dataframe
print("\nNúmero de contratos:")
contracts_panel_df.info()
print("\nAlquiler medio:")
avg_rent_df.info()
print("\nAlquiler medio por m2:")
avg_rent_m2_df.info()
print("\nSuperficie media:")
avg_surface_df.info()


Número de contratos:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3504 entries, 0 to 3503
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   neighborhood_code  3504 non-null   int64  
 1   neighborhood       3504 non-null   object 
 2   year               3504 non-null   int64  
 3   quarter            3504 non-null   int64  
 4   num_contracts      3429 non-null   float64
dtypes: float64(1), int64(3), object(1)
memory usage: 137.0+ KB

Alquiler medio:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3504 entries, 0 to 3503
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   neighborhood_code  3504 non-null   int64  
 1   neighborhood       3504 non-null   object 
 2   year               3504 non-null   int64  
 3   quarter            3504 non-null   int64  
 4   avg_rent           3283 non-null   float64
dtypes: float

In [48]:
avg_surface_df.head()

,neighborhood_code,neighborhood,year,quarter,avg_surface
0,1,el Raval,2014,1,60.44663
1,1,el Raval,2014,2,57.37912
2,1,el Raval,2014,3,65.05985
3,1,el Raval,2014,4,59.96962
4,2,el Barri Gòtic,2014,1,78.13333


# validation

In [44]:

validate_panel_df(avg_rent_df, "avg_rent")


========== VALIDACIÓN PANEL: avg_rent ==========

1. Duplicados por (neighborhood_code, year, quarter):
   Total duplicados: 0

2. Combinaciones esperadas vs observadas:
   Esperadas: 3504
   Observadas: 3504
   Faltantes: 0

3. Missing avg_rent por year-quarter:
 year  quarter  missing_count
 2014        1              5
 2014        2              3
 2014        3              5
 2014        4              6
 2015        1              6
 2015        2              2
 2015        3              4
 2015        4              4
 2016        1              2
 2016        2              5
 2016        3              4
 2016        4              4
 2017        1              0
 2017        2              0
 2017        3              0
 2017        4              0
 2018        1              6
 2018        2              5
 2018        3              6
 2018        4              5
 2019        1              4
 2019        2              6
 2019        3              6
 2019        4  

In [45]:
validate_panel_df(avg_rent_m2_df, "avg_rent_m2")


========== VALIDACIÓN PANEL: avg_rent_m2 ==========

1. Duplicados por (neighborhood_code, year, quarter):
   Total duplicados: 0

2. Combinaciones esperadas vs observadas:
   Esperadas: 3504
   Observadas: 3504
   Faltantes: 0

3. Missing avg_rent_m2 por year-quarter:
 year  quarter  missing_count
 2014        1              5
 2014        2              3
 2014        3              5
 2014        4              6
 2015        1              6
 2015        2              2
 2015        3              4
 2015        4              4
 2016        1              2
 2016        2              5
 2016        3              5
 2016        4              4
 2017        1              0
 2017        2              0
 2017        3              0
 2017        4              0
 2018        1              6
 2018        2              5
 2018        3              6
 2018        4              5
 2019        1              4
 2019        2              6
 2019        3              5
 2019     

In [46]:
validate_panel_df(avg_surface_df, "avg_surface")

========== VALIDACIÓN PANEL: avg_surface ==========

1. Duplicados por (neighborhood_code, year, quarter):
   Total duplicados: 0

2. Combinaciones esperadas vs observadas:
   Esperadas: 3504
   Observadas: 3504
   Faltantes: 0

3. Missing avg_surface por year-quarter:
 year  quarter  missing_count
 2014        1              0
 2014        2              1
 2014        3              2
 2014        4              1
 2015        1              0
 2015        2              1
 2015        3              0
 2015        4              0
 2016        1              0
 2016        2              0
 2016        3              0
 2016        4              1
 2017        1              1
 2017        2              0
 2017        3              0
 2017        4              0
 2018        1              2
 2018        2              2
 2018        3              1
 2018        4              0
 2019        1              2
 2019        2              0
 2019        3              1
 2019     

# Merge de los 4 dataframes

In [49]:
import pandas as pd


# =========================================================
# 1. CHEQUEO OPCIONAL: consistencia de nombres de barrio
# =========================================================

def compare_neighborhood_names(base_df, other_df, other_name="other_df"):
    merged_names = base_df[
        ["neighborhood_code", "year", "quarter", "neighborhood"]
    ].merge(
        other_df[["neighborhood_code", "year", "quarter", "neighborhood"]],
        on=["neighborhood_code", "year", "quarter"],
        how="inner",
        suffixes=("_base", f"_{other_name}")
    )

    mismatches = merged_names[
        merged_names["neighborhood_base"] != merged_names[f"neighborhood_{other_name}"]
    ]

    print(f"\nComparación de nombres contra {other_name}:")
    print("Total filas comparadas:", len(merged_names))
    print("Total diferencias de nombre:", len(mismatches))

    if len(mismatches) > 0:
        print(mismatches.head(10))


In [50]:
compare_neighborhood_names(contracts_panel_df, avg_rent_df, "avg_rent")
compare_neighborhood_names(contracts_panel_df, avg_rent_m2_df, "avg_rent_m2")
compare_neighborhood_names(contracts_panel_df, avg_surface_df, "avg_surface")


Comparación de nombres contra avg_rent:
Total filas comparadas: 3504
Total diferencias de nombre: 0

Comparación de nombres contra avg_rent_m2:
Total filas comparadas: 3504
Total diferencias de nombre: 0

Comparación de nombres contra avg_surface:
Total filas comparadas: 3504
Total diferencias de nombre: 0


# merge final


In [51]:

# 2. MERGE FINAL DEL PANEL


rental_panel_df = (
    contracts_panel_df
    .merge(
        avg_rent_df[["neighborhood_code", "year", "quarter", "avg_rent"]],
        on=["neighborhood_code", "year", "quarter"],
        how="left",
        validate="one_to_one"
    )
    .merge(
        avg_rent_m2_df[["neighborhood_code", "year", "quarter", "avg_rent_m2"]],
        on=["neighborhood_code", "year", "quarter"],
        how="left",
        validate="one_to_one"
    )
    .merge(
        avg_surface_df[["neighborhood_code", "year", "quarter", "avg_surface"]],
        on=["neighborhood_code", "year", "quarter"],
        how="left",
        validate="one_to_one"
    )
)



In [52]:
# ordenar y reordenar columnas
rental_panel_df = rental_panel_df[
    [
        "neighborhood_code",
        "neighborhood",
        "year",
        "quarter",
        "num_contracts",
        "avg_rent",
        "avg_rent_m2",
        "avg_surface",
    ]
].sort_values(["year", "neighborhood_code", "quarter"]).reset_index(drop=True)

In [54]:
rental_panel_df.head()

,neighborhood_code,neighborhood,year,quarter,num_contracts,avg_rent,avg_rent_m2,avg_surface
0,1,el Raval,2014,1,356.0,589.548624,10.76,60.44663
1,1,el Raval,2014,2,409.0,550.631711,10.52,57.37912
2,1,el Raval,2014,3,423.0,576.454232,9.84,65.05985
3,1,el Raval,2014,4,395.0,596.995494,10.81,59.96962
4,2,el Barri Gòtic,2014,1,135.0,712.786519,10.58,78.13333


In [55]:
rental_panel_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3504 entries, 0 to 3503
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   neighborhood_code  3504 non-null   int64  
 1   neighborhood       3504 non-null   object 
 2   year               3504 non-null   int64  
 3   quarter            3504 non-null   int64  
 4   num_contracts      3429 non-null   float64
 5   avg_rent           3283 non-null   float64
 6   avg_rent_m2        3285 non-null   float64
 7   avg_surface        3400 non-null   float64
dtypes: float64(4), int64(3), object(1)
memory usage: 219.1+ KB


In [56]:
rental_panel_df["num_contracts"] = rental_panel_df["num_contracts"].astype("Int64")

In [57]:
rental_panel_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3504 entries, 0 to 3503
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   neighborhood_code  3504 non-null   int64  
 1   neighborhood       3504 non-null   object 
 2   year               3504 non-null   int64  
 3   quarter            3504 non-null   int64  
 4   num_contracts      3429 non-null   Int64  
 5   avg_rent           3283 non-null   float64
 6   avg_rent_m2        3285 non-null   float64
 7   avg_surface        3400 non-null   float64
dtypes: Int64(1), float64(3), int64(3), object(1)
memory usage: 222.6+ KB


# guardar a parquet

In [58]:
rental_panel_df.to_parquet("rental_panel_barris_2014_2025.parquet", index=False)

In [59]:
df = pd.read_parquet("rental_panel_barris_2014_2025.parquet")
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3504 entries, 0 to 3503
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   neighborhood_code  3504 non-null   int64  
 1   neighborhood       3504 non-null   object 
 2   year               3504 non-null   int64  
 3   quarter            3504 non-null   int64  
 4   num_contracts      3429 non-null   Int64  
 5   avg_rent           3283 non-null   float64
 6   avg_rent_m2        3285 non-null   float64
 7   avg_surface        3400 non-null   float64
dtypes: Int64(1), float64(3), int64(3), object(1)
memory usage: 222.6+ KB
None


# guardar a CSV para copia legible

In [60]:
rental_panel_df.to_csv("rental_panel_barris_2014_2025.csv", index=False, encoding="utf-8-sig")

# Extaccion por distritos

In [71]:
# =========================================================
# 1. UTILIDADES
# =========================================================

def clean_text(value):
    if value is None:
        return ""
    return " ".join(str(value).strip().split())


def parse_nullable_number(value, number_type="float"):
    """
    Convierte a int o float.
    Si viene vacío, 'n.d.' u otro valor no numérico, devuelve None.
    """
    if value is None:
        return None

    number_type = str(number_type).strip().lower()
    if number_type in {"int", "integer"}:
        number_type = "int"
    elif number_type in {"float", "double", "decimal"}:
        number_type = "float"
    else:
        raise ValueError("number_type debe ser 'int' o 'float'")

    if isinstance(value, (int, float)):
        return int(value) if number_type == "int" else float(value)

    text = str(value).strip()

    if text == "":
        return None

    if text.lower() in {"n.d.", "nd", "n.d", "na", "n/a", "-"}:
        return None

    text = text.replace(",", ".")

    try:
        return int(float(text)) if number_type == "int" else float(text)
    except ValueError:
        return None


def get_year_sheets(filepath, start_year=2000, end_year=2025):
    wb = load_workbook(filepath, data_only=True)

    valid_sheets = []
    for sheet_name in wb.sheetnames:
        if str(sheet_name).isdigit():
            year = int(sheet_name)
            if start_year <= year <= end_year:
                valid_sheets.append(sheet_name)

    return sorted(valid_sheets, key=int)


def parse_old_district_label(text):
    """
    Convierte '1. Ciutat Vella' -> (1, 'Ciutat Vella')
    """
    text = clean_text(text)
    match = re.match(r"^(\d+)\.\s*(.+)$", text)
    if not match:
        return None, None

    district_code = int(match.group(1))
    district_name = match.group(2).strip()
    return district_code, district_name


# =========================================================
# 2. FORMATO NUEVO (2014–2025)
# =========================================================

def extract_districts_new_format(rows, year, value_col_name, number_type="float"):
    """
    Formato 2014+:
    col 0 = code
    col 1 = district
    col 2..5 = I..IV
    """
    district_start_idx = None

    for i, row in enumerate(rows):
        second_cell = clean_text(row[1]) if len(row) > 1 else ""
        if second_cell.startswith("Districtes municipals"):
            district_start_idx = i
            break

    if district_start_idx is None:
        raise ValueError(f"No encontré la sección 'Districtes municipals' para {year}.")

    quarter_cols = {
        2: 1,
        3: 2,
        4: 3,
        5: 4,
    }

    result = []

    for i in range(district_start_idx + 1, len(rows)):
        row = rows[i]

        code = row[0] if len(row) > 0 else None
        district = clean_text(row[1]) if len(row) > 1 else ""

        if code is None or district == "":
            break

        if not isinstance(code, int):
            break

        for col_idx, quarter in quarter_cols.items():
            value = row[col_idx] if col_idx < len(row) else None

            result.append({
                "district_code": code,
                "district": district,
                "year": year,
                "quarter": quarter,
                value_col_name: parse_nullable_number(value, number_type=number_type),
            })

    return result


# =========================================================
# 3. FORMATO VIEJO (2000–2013)
# =========================================================

def extract_districts_old_format(rows, year, value_col_name, number_type="float"):
    """
    Formato 2000–2013:
    col 0 = '1. Ciutat Vella'
    col 1..4 = I..IV
    fila final 'Barcelona' se excluye
    """
    quarter_cols = {
        1: 1,
        2: 2,
        3: 3,
        4: 4,
    }

    result = []

    for row in rows:
        first_cell = clean_text(row[0]) if len(row) > 0 else ""

        if first_cell == "":
            continue

        if first_cell == "Barcelona":
            break

        district_code, district = parse_old_district_label(first_cell)

        if district_code is None:
            continue

        for col_idx, quarter in quarter_cols.items():
            value = row[col_idx] if col_idx < len(row) else None

            result.append({
                "district_code": district_code,
                "district": district,
                "year": year,
                "quarter": quarter,
                value_col_name: parse_nullable_number(value, number_type=number_type),
            })

    return result


# =========================================================
# 4. EXTRACCIÓN GENÉRICA
# =========================================================

def extract_districts_one_sheet_df(filepath, sheet_name, value_col_name, number_type="float"):
    wb = load_workbook(filepath, data_only=True)
    ws = wb[sheet_name]
    rows = list(ws.iter_rows(values_only=True))

    year = int(sheet_name)

    if year >= 2014:
        result = extract_districts_new_format(
            rows=rows,
            year=year,
            value_col_name=value_col_name,
            number_type=number_type
        )
    else:
        result = extract_districts_old_format(
            rows=rows,
            year=year,
            value_col_name=value_col_name,
            number_type=number_type
        )

    df = pd.DataFrame(result)
    df = df.sort_values(["year", "district_code", "quarter"]).reset_index(drop=True)
    return df


def extract_districts_all_years_df(
    filepath,
    value_col_name,
    number_type="float",
    start_year=2000,
    end_year=2025
):
    year_sheets = get_year_sheets(filepath, start_year=start_year, end_year=end_year)

    dfs = []
    for sheet_name in year_sheets:
        df_sheet = extract_districts_one_sheet_df(
            filepath=filepath,
            sheet_name=sheet_name,
            value_col_name=value_col_name,
            number_type=number_type
        )
        dfs.append(df_sheet)

    df = pd.concat(dfs, ignore_index=True)
    df = df.sort_values(["year", "district_code", "quarter"]).reset_index(drop=True)
    return df

In [72]:
district_contracts_df = extract_districts_all_years_df(
    filepath="trimestral_bcn_contractes.xlsx",
    value_col_name="num_contracts",
    number_type="int",
    start_year=2000,
    end_year=2025
)

In [73]:
district_contracts_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1040 entries, 0 to 1039
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   district_code  1040 non-null   int64  
 1   district       1040 non-null   object 
 2   year           1040 non-null   int64  
 3   quarter        1040 non-null   int64  
 4   num_contracts  1030 non-null   float64
dtypes: float64(1), int64(3), object(1)
memory usage: 40.8+ KB


In [76]:
print(district_contracts_df.head(12))
print(district_contracts_df.tail(12))
print(district_contracts_df.shape)

    district_code        district  year  quarter  num_contracts
0               1    Ciutat Vella  2000        1          397.0
1               1    Ciutat Vella  2000        2          355.0
2               1    Ciutat Vella  2000        3          373.0
3               1    Ciutat Vella  2000        4          415.0
4               2        Eixample  2000        1         1200.0
5               2        Eixample  2000        2         1082.0
6               2        Eixample  2000        3          832.0
7               2        Eixample  2000        4         1093.0
8               3  Sants-Montjuïc  2000        1          427.0
9               3  Sants-Montjuïc  2000        2          365.0
10              3  Sants-Montjuïc  2000        3          371.0
11              3  Sants-Montjuïc  2000        4          413.0
      district_code     district  year  quarter  num_contracts
1028              8   Nou Barris  2025        1          591.0
1029              8   Nou Barris  2025    

In [75]:
#inspect missing values
validate_district_panel_df(district_contracts_df, "num_contracts")

========== VALIDACIÓN DISTRITOS: num_contracts ==========

Shape: (1040, 5)
Duplicados por clave: 0

Missing por columna:
district_code     0
district          0
year              0
quarter           0
num_contracts    10
dtype: int64

Registros por año:
year
2000    40
2001    40
2002    40
2003    40
2004    40
2005    40
2006    40
2007    40
2008    40
2009    40
2010    40
2011    40
2012    40
2013    40
2014    40
2015    40
2016    40
2017    40
2018    40
2019    40
2020    40
2021    40
2022    40
2023    40
2024    40
2025    40
dtype: int64

Missing por year-quarter:
 year  quarter  missing_count
 2000        1              0
 2000        2              0
 2000        3              0
 2000        4              0
 2001        1              0
 2001        2              0
 2001        3              0
 2001        4              0
 2002        1              0
 2002        2              0
 2002        3              0
 2002        4              0
 2003        1          

In [77]:
# 2) Alquiler medio
district_avg_rent_df = extract_districts_all_years_df(
    filepath="trimestral_bcn_lloguer.xlsx",
    value_col_name="avg_rent",
    number_type="float",
    start_year=2000,
    end_year=2025
)

# 3) Alquiler medio por m2
district_avg_rent_m2_df = extract_districts_all_years_df(
    filepath="trimestral_bcn_lloguer_m2.xlsx",
    value_col_name="avg_rent_m2",
    number_type="float",
    start_year=2000,
    end_year=2025
)

# 4) Superficie media
district_avg_surface_df = extract_districts_all_years_df(
    filepath="trimestral_bcn_sup.xlsx",
    value_col_name="avg_surface",
    number_type="float",
    start_year=2000,
    end_year=2025
)

In [78]:
district_contracts_df["num_contracts"] = district_contracts_df["num_contracts"].astype("Int64")

In [79]:
print("Número de contratos:")
print(district_contracts_df.info())

print("\nAlquiler medio:")
print(district_avg_rent_df.info())

print("\nAlquiler medio por m2:")
print(district_avg_rent_m2_df.info())

print("\nSuperficie media:")
print(district_avg_surface_df.info())

Número de contratos:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1040 entries, 0 to 1039
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   district_code  1040 non-null   int64 
 1   district       1040 non-null   object
 2   year           1040 non-null   int64 
 3   quarter        1040 non-null   int64 
 4   num_contracts  1030 non-null   Int64 
dtypes: Int64(1), int64(3), object(1)
memory usage: 41.8+ KB
None

Alquiler medio:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1040 entries, 0 to 1039
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   district_code  1040 non-null   int64  
 1   district       1040 non-null   object 
 2   year           1040 non-null   int64  
 3   quarter        1040 non-null   int64  
 4   avg_rent       1030 non-null   float64
dtypes: float64(1), int64(3), object(1)
memory usage: 40.8+ KB
None

Alquil

# Merge district

In [80]:
district_panel_df = (
    district_contracts_df
    .merge(
        district_avg_rent_df[["district_code", "year", "quarter", "avg_rent"]],
        on=["district_code", "year", "quarter"],
        how="left",
        validate="one_to_one"
    )
    .merge(
        district_avg_rent_m2_df[["district_code", "year", "quarter", "avg_rent_m2"]],
        on=["district_code", "year", "quarter"],
        how="left",
        validate="one_to_one"
    )
    .merge(
        district_avg_surface_df[["district_code", "year", "quarter", "avg_surface"]],
        on=["district_code", "year", "quarter"],
        how="left",
        validate="one_to_one"
    )
)

district_panel_df = district_panel_df[
    [
        "district_code",
        "district",
        "year",
        "quarter",
        "num_contracts",
        "avg_rent",
        "avg_rent_m2",
        "avg_surface",
    ]
].sort_values(["year", "district_code", "quarter"]).reset_index(drop=True)

In [82]:
district_panel_df.info()    

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1040 entries, 0 to 1039
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   district_code  1040 non-null   int64  
 1   district       1040 non-null   object 
 2   year           1040 non-null   int64  
 3   quarter        1040 non-null   int64  
 4   num_contracts  1030 non-null   Int64  
 5   avg_rent       1030 non-null   float64
 6   avg_rent_m2    1030 non-null   float64
 7   avg_surface    1030 non-null   float64
dtypes: Int64(1), float64(3), int64(3), object(1)
memory usage: 66.1+ KB


In [84]:
print(district_panel_df[district_panel_df.isna().any(axis=1)].to_string(index=False))

 district_code            district  year  quarter  num_contracts  avg_rent  avg_rent_m2  avg_surface
             1        Ciutat Vella  2025        4           <NA>       NaN          NaN          NaN
             2            Eixample  2025        4           <NA>       NaN          NaN          NaN
             3      Sants-Montjuïc  2025        4           <NA>       NaN          NaN          NaN
             4           Les Corts  2025        4           <NA>       NaN          NaN          NaN
             5 Sarrià-Sant Gervasi  2025        4           <NA>       NaN          NaN          NaN
             6              Gràcia  2025        4           <NA>       NaN          NaN          NaN
             7      Horta-Guinardó  2025        4           <NA>       NaN          NaN          NaN
             8          Nou Barris  2025        4           <NA>       NaN          NaN          NaN
             9         Sant Andreu  2025        4           <NA>       NaN          NaN    

# Guardar por districto



In [85]:
district_panel_df.to_parquet("district_panel_2000_2025.parquet", index=False)
district_panel_df.to_csv("district_panel_2000_2025.csv", index=False, encoding="utf-8-sig")